# Companion Notebook 02: RMSNorm and SwiGLU from Scratch in PyTorch

This notebook implements **Root Mean Square Normalization (RMSNorm)** and the **Swish-Gated Linear Unit (SwiGLU)** activation layer from scratch in PyTorch, verifying outputs against hand-calculations.


## 1. RMSNorm Implementation
We implement the RMSNorm module and test it with a sample input tensor $\mathbf{x} = [2.0, -1.0, 3.0]$.


In [1]:
import torch
import torch.nn as nn
import numpy as np

# Set print option for clean tensor printing
torch.set_printoptions(precision=4, sci_mode=False)

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super(RMSNorm, self).__init__()
        self.eps = eps
        # Learnable scale parameters (gamma)
        self.weight = nn.Parameter(torch.ones(dim))
        
    def forward(self, x):
        # x shape: [batch_size, seq_len, dim] or [batch_size, dim]
        # Calculate root mean square along the hidden dimension
        variance = x.pow(2).mean(-1, keepdim=True)
        rms = torch.sqrt(variance + self.eps)
        # Normalize and scale
        return (x / rms) * self.weight

# Verify with hand-calculation input
x_ln = torch.tensor([[2.0, -1.0, 3.0]])
norm_layer = RMSNorm(dim=3)

# Print initial weights (gamma initialized to 1s)
print("Initial Gamma Weight:", norm_layer.weight.data.numpy())

# Run forward pass
output_ln = norm_layer(x_ln)
print("\nRMSNorm Input Tensor x:\n", x_ln.numpy())
print("RMSNorm Output Tensor:\n", output_ln.detach().numpy())


Initial Gamma Weight: [1. 1. 1.]

RMSNorm Input Tensor x:
 [[ 2. -1.  3.]]
RMSNorm Output Tensor:
 [[ 0.9258191  -0.46290955  1.3887286 ]]


### Explanation of Outputs (RMSNorm)
- **Initial Gamma Weight**: $[1.0, 1.0, 1.0]$ initializes scaling factors as identities.
- **RMSNorm Output Tensor**: Input tensor $[2.0, -1.0, 3.0]$ is normalized by its root mean square value $\approx 2.1603$ yielding output $[0.9258, -0.4629, 1.3887]$, matching the hand calculation in the study guide exactly to four decimal places.


## 2. SwiGLU Gating Implementation
We implement SwiGLU using a single layer split or three custom linear operations, and test it with target weight matrices loaded directly into PyTorch parameters.


In [2]:
class SwiGLU(nn.Module):
    def __init__(self, in_dim, out_dim, intermediate_dim):
        super(SwiGLU, self).__init__()
        # W1: Gate projection
        self.w1 = nn.Linear(in_dim, intermediate_dim, bias=False)
        # W2: Value projection
        self.w2 = nn.Linear(in_dim, intermediate_dim, bias=False)
        # W3: Output projection
        self.w3 = nn.Linear(intermediate_dim, out_dim, bias=False)
        
    def forward(self, x):
        # Swish(x * W1) * (x * W2)
        gate = self.w1(x)
        value = self.w2(x)
        # Swish is x * sigmoid(x)
        swish_gate = gate * torch.sigmoid(gate)
        # Hadamard product gated by Swish
        gated = swish_gate * value
        # Project output
        return self.w3(gated)

# Verify with hand-calculation values
x_swiglu = torch.tensor([[0.5, 0.5]])
swiglu_layer = SwiGLU(in_dim=2, out_dim=1, intermediate_dim=1)

# Manually load the weights from the hand calculation
# PyTorch nn.Linear expects weights shaped [out_features, in_features]
swiglu_layer.w1.weight.data = torch.tensor([[2.0, -2.0]])
swiglu_layer.w2.weight.data = torch.tensor([[-1.0, 3.0]])
swiglu_layer.w3.weight.data = torch.tensor([[1.5]])

print("Loaded W1 weight:\n", swiglu_layer.w1.weight.data.numpy())
print("Loaded W2 weight:\n", swiglu_layer.w2.weight.data.numpy())
print("Loaded W3 weight:\n", swiglu_layer.w3.weight.data.numpy())

# Run forward pass
output_swiglu = swiglu_layer(x_swiglu)
print("\nSwiGLU Input Tensor x:\n", x_swiglu.numpy())
print("SwiGLU Output Tensor:\n", output_swiglu.detach().numpy())


Loaded W1 weight:
 [[ 2. -2.]]
Loaded W2 weight:
 [[-1.  3.]]
Loaded W3 weight:
 [[1.5]]

SwiGLU Input Tensor x:
 [[0.5 0.5]]
SwiGLU Output Tensor:
 [[0.]]


### Explanation of Outputs (SwiGLU)
- **Loaded weights**: $W_1, W_2, W_3$ represent matrices from the hand calculation.
- **SwiGLU Output Tensor**: With input $[0.5, 0.5]$, the Swish gate output evaluates to exactly $0.0$, multiplying value projections to yield $0.0$, matching our gating calculation.


## 3. Training Causal Masking
In training mode, target masks enforce that token $i$ cannot attend to tokens $> i$. We show how this mask matrix is generated.


In [3]:
# Sequence length of 4
seq_len = 4
# Create upper triangular mask with float('-inf') on non-allowed positions
mask = torch.triu(torch.full((seq_len, seq_len), float('-inf')), diagonal=1)
print("Causal Attention Mask Matrix:")
print(mask.numpy())


Causal Attention Mask Matrix:
[[  0. -inf -inf -inf]
 [  0.   0. -inf -inf]
 [  0.   0.   0. -inf]
 [  0.   0.   0.   0.]]


### Explanation of Outputs (Causal Attention Mask)
- **Mask Matrix**: Contains $0.0$ on allowed query positions and $-\infty$ on future positions (above main diagonal), preventing tokens from attending to subsequent tokens during auto-regressive generation.
